In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, FloatType
from delta.tables import DeltaTable

%md
### Functions of Data Cleaning and Transformation for Dimension Tables

In [0]:
def transform_brands(df):
    anomalies = {
        "GROCERY": "GRCY",
        "ELECTRONICS": "CE",
        "APPAREL": "APP",
        "HOME & KITCHEN": "HNK",
        "BEAUTY & PERSONAL CARE": "BPC",
        "SPORTS & OUTDOORS": "SPT",
        "BOOKS": "BKS",
        "TOYS": "TOY"
    }
    return (df
        .withColumn("brand_name", F.trim(F.col("brand_name")))
        .withColumn("brand_code", F.regexp_replace(F.col("brand_code"), r'[^A-Za-z0-9]', ''))
        .replace(to_replace=anomalies, subset=["category_code"])
    )

def transform_category(df):
    return (df
        .dropDuplicates(['category_code'])
        .withColumn("category_code", F.upper(F.col("category_code")))
    )

def transform_products(df):
    return (df
        .withColumn("weight_grams", F.regexp_replace(F.col("weight_grams"), "g", "").cast(IntegerType()))
        .withColumn("length_cm", F.regexp_replace(F.col("length_cm"), ",", ".").cast(FloatType()))
        .withColumn("category_code", F.upper(F.col("category_code")))
        .withColumn("brand_code", F.upper(F.col("brand_code")))
        .withColumn("material", 
            F.when(F.col("material") == "Coton", "Cotton")
             .when(F.col("material") == "Alumium", "Aluminum")
             .when(F.col("material") == "Ruber", "Rubber")
             .otherwise(F.col("material"))
        )
        .withColumn("rating_count", 
            F.when(F.col("rating_count").isNotNull(), F.abs(F.col("rating_count")))
             .otherwise(F.lit(0))
        )
    )

def transform_customers(df):
    return (df
        .dropna(subset=["customer_id"])
        .fillna("Not Available", subset=["phone"])
    )

def transform_date(df):
    return (df
        .withColumn("date", F.to_date(F.col("date"), "dd-MM-yyyy"))
        .dropDuplicates(['date'])
        .withColumn("day_name", F.initcap(F.col("day_name")))
        .withColumn("week_of_year", F.abs(F.col("week_of_year")))
        .withColumn("quarter", F.concat(F.lit("Q"), F.col("quarter"), F.lit("-"), F.col("year")))
        .withColumn("week", F.concat(F.lit("Week-"), F.col("week_of_year"), F.lit("-"), F.col("year")))
        .drop("week_of_year")
    )

### Map Table Name -> Transformation Function

In [0]:
TRANSFORMATION_MAP = {
    'brands': transform_brands,
    'category': transform_category,
    'products': transform_products,
    'customers': transform_customers,
    'date': transform_date
}

### Bronze to Silver Dynamic Pipeline Engine:

In [0]:
def sync_dimension_silver(table_name, primary_key):
    silver_table_name = f"ecommerce.silver.slv_{table_name}"
    
    # Step 1: Read Bronze Table
    df_bronze_dim = spark.table(f"ecommerce.bronze.brz_{table_name}")
    
    # Step 2: Get Latest Snapshot Based on Ingestion Timestamp
    max_ingest_time = df_bronze_dim.select(F.max("_ingested_at")).collect()[0][0]
    
    if max_ingest_time is None:
        print(f"No records found in Bronze layer for {table_name}.")
        return

    df_latest_source = (df_bronze_dim
        .filter(F.col("_ingested_at") == max_ingest_time)
        .dropDuplicates([primary_key])
    )

    # Step 3: Apply Specific Transformation Rule Dynamically
    if table_name in TRANSFORMATION_MAP:
        df_cleaned = TRANSFORMATION_MAP[table_name](df_latest_source)
    else:
        df_cleaned = df_latest_source

    # Step 4: Add Audit Metadata
    df_final = df_cleaned.withColumn("_silver_processed_at", F.current_timestamp())

    # Step 5: Save or Upsert to Silver Layer
    if not spark.catalog.tableExists(silver_table_name):
        # FIRST TIME: Create Silver Delta Table
        (df_final.write
            .format("delta")
            .mode("overwrite")
            .option("mergeSchema", "true")
            .saveAsTable(silver_table_name)
        )
        print(f"Successfully created initial Silver table: {silver_table_name}")
    else:
        # SUBSEQUENT RUNS: Merge + Automatic Stale Data Removal
        target_silver = DeltaTable.forName(spark, silver_table_name)

        (target_silver.alias("target")
            .merge(
                df_final.alias("source"),
                f"target.{primary_key} = source.{primary_key}"
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .whenNotMatchedBySourceDelete()
            .execute()
        )
        print(f"Successfully cleaned and updated Silver table: {silver_table_name}")

### Execute Dimension Tables

In [0]:
dim_tables_config = {
    'customers': 'customer_id',
    'brands': 'brand_id',
    'category': 'category_id',
    'products': 'product_id',
    'date': 'date'
}

for table, pk in dim_tables_config.items():
    sync_dimension_silver(table, pk)

Successfully created initial Silver table: ecommerce.silver.slv_customers
Successfully created initial Silver table: ecommerce.silver.slv_brands
Successfully created initial Silver table: ecommerce.silver.slv_category
Successfully created initial Silver table: ecommerce.silver.slv_products
